# Z-Image

https://huggingface.co/Tongyi-MAI/Z-Image-Turbo

In [ ]:
import torch
from diffusers import ZImagePipeline

model_id = "Tongyi-MAI/Z-Image-Turbo"

pipe = ZImagePipeline.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=False,
)
pipe.to("cuda")

## disable flash attention since compilation of flash-attn during 'uv sync' fails
# pipe.transformer.set_attention_backend("flash")
# pipe.transformer.set_attention_backend("_flash_3")

# compile to DiT model to accelerate inference
pipe.transformer.compile()

generate_kwargs = {
    "height": 1024,
    "width": 1024,
    "guidance_scale": 0.0, # guidance should be 0 for turbo models (according to the model card)
    "num_inference_steps": 9,
    "max_sequence_length": 512,
}

In [ ]:
prompt = "A cat holding a sign that says hello world"

image = pipe(
    prompt=prompt,
    **generate_kwargs,
    generator=torch.Generator("cuda").manual_seed(42),
).images[0]

image

In [ ]:
prompt = "Young Chinese woman in red Hanfu, intricate embroidery. Impeccable makeup, red floral forehead pattern. Elaborate high bun, golden phoenix headdress, red flowers, beads. Holds round folding fan with lady, trees, bird. Neon lightning-bolt lamp (⚡️), bright yellow glow, above extended left palm. Soft-lit outdoor night background, silhouetted tiered pagoda (西安大雁塔), blurred colorful distant lights."

# 2. Generate Image
image = pipe(
    prompt=prompt,
    **generate_kwargs,
    generator=torch.Generator("cuda").manual_seed(42),
).images[0]

image